# MatterGen Colab Tutorial for Computational Chemistry Students

> **Goal:** Learn how to install MatterGen in Google Colab, understand the theory and architecture, run unconditional and property-conditioned generation, and visualize generated inorganic crystal structures.

This notebook is intentionally **verbose** and is designed as a teaching resource for students who are new to generative models in materials science.


## 0. How to use this notebook

- If you are in **Google Colab**, select **Runtime → Change runtime type → GPU**.
- Run cells in order from top to bottom.
- Some generation steps can take several minutes depending on GPU type.

### What you will learn
1. The methodological context of MatterGen (why diffusion for crystals).
2. The high-level architecture and training recipe.
3. Practical generation workflows (unconditional and conditioned).
4. How to inspect, parse, and visualize outputs.


## 1. Install prerequisites (Colab-friendly)

This section installs MatterGen and companion libraries for analysis/visualization.


In [ ]:
# @title
# 1. Clone the repository (if not already present)
import os
if not os.path.exists('mattergen'):
    !git clone https://github.com/JoshDumo/mattergen.git

# 2. Change directory using %cd magic
%cd mattergen

# 3. Setup python env and install MatterGen and deps with uv
!mkdir /backend-container
!mkdir /backend-container/containers
!touch /backend-container/containers/build.constraints
!touch /backend-container/containers/requirements.constraints
!pip install uv
!uv venv .venv
!source .venv/bin/activate; uv pip install -e .

# 4. Install cuda toolkit
!apt-get install nvidia-cuda-toolkit

Cloning into 'mattergen'...
remote: Enumerating objects: 1642, done.
remote: Counting objects: 100% (395/395), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 1642 (delta 292), reused 238 (delta 238), pack-reused 1247 (from 2)
Receiving objects: 100% (1642/1642), 3.38 MiB | 19.55 MiB/s, done.
Resolving deltas: 100% (643/643), done.
/content/mattergen
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 40.7 MB/s eta 0:00:00
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Using Python 3.12.13 environment at: /usr
Resolved 248 packages in 2.82s
Prepared 96 packages in 51.29s
Uninstalled 6 packages in 637ms
Installed 96 packages in 933ms
 + aioitertools==0.13.0
 + ase==3.25.0
 + astroid==4.0.4
 + async-lru==2.3.0
 + atomate2==0.0.18
 + autopep8==2.3.2
 + azure-core==1.39.0
 + azure-identity==1.25.3
 + azure-storage-blob==12.28.0
 + bcrypt==5.0.0
 + blake3==1.0.8
 + boto3==1.42

In [ ]:
# @title
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.12.13
PyTorch: 2.2.1+cu118
CUDA available: True
GPU: Tesla T4


## 2. Methodology and theory (teaching notes)

### 2.1 Why generative modeling for computational chemistry?
Classical crystal discovery workflows (enumeration, substitution heuristics, random structure search, and high-throughput DFT) are powerful but expensive. We want a model that can:

- Learn structural priors from known inorganic crystals.
- Propose *novel* candidate structures with plausible chemistry.
- Be steerable toward target properties (e.g., band gap, magnetic density).

### 2.2 Why diffusion models?
MatterGen uses a diffusion-style approach where generation is treated as iterative denoising from noise to a crystal representation. Conceptually:

1. **Forward process:** progressively corrupts a training crystal with noise.
2. **Reverse process:** neural network learns to remove that noise step-by-step.
3. **Sampling:** start from noise at inference time and apply learned reverse dynamics.

Diffusion models are attractive because they are generally stable to train, scale well, and support guidance/conditioning strategies.

### 2.3 Crystal-specific representation challenge
Crystals are not simple fixed-length vectors: they involve

- Atomic species (discrete)
- Atomic positions (continuous, periodic)
- Lattice geometry (continuous, constrained)

MatterGen's architecture combines geometric deep learning ideas with diffusion modeling so these components can be denoised coherently.

### 2.4 Conditioning and classifier-free guidance intuition
For property-conditioned generation, MatterGen can be trained/fine-tuned with condition signals. During sampling, a guidance scale (diffusion guidance factor) can trade off:

- **Higher guidance:** better property adherence
- **Lower guidance:** more diversity/realism

This is analogous to classifier-free guidance in text-to-image diffusion, but adapted to materials property conditioning.

### 2.5 How training works (high-level)
At a high level, training includes:

1. Sample a crystal structure from a training dataset.
2. Corrupt it at a random timestep.
3. Predict denoising targets (for positions/lattice/species related terms).
4. Optimize a weighted diffusion loss over many timesteps.
5. For conditioned variants, include property embeddings and conditional dropout for guidance.

In practice, the repository contains both base and fine-tuning workflows as scripts/configs; this notebook focuses on **usage** and **scientific interpretation** rather than reproducing full large-scale pretraining.


## 3. MatterGen architecture map (code-oriented orientation)

When you later inspect the repository, these modules are useful landmarks:

- `mattergen/diffusion/`: diffusion mechanics, losses, modules.
- `mattergen/generator.py`: sampling/generation orchestration.
- `mattergen/property_embeddings.py`: handling conditioning/property embeddings.
- `mattergen/scripts/generate.py`: CLI entrypoint for generation.
- `mattergen/scripts/finetune.py`: fine-tuning workflow for conditional models.

As a student exercise, try opening these files side-by-side and mapping each concept in Section 2 to implementation components.


## 4. Unconditional generation with a pre-trained checkpoint

We first generate candidate structures with the base model (`mattergen_base`).


In [ ]:
from pathlib import Path

# Define output directory
results_dir = Path("/content/results_unconditional")
results_dir.mkdir(exist_ok=True)

print("Running generation...")
!source .venv/bin/activate; export MODEL_NAME=mattergen_base ; export RESULTS_PATH=/content/results_unconditional/ ; mattergen-generate $RESULTS_PATH --pretrained-name=$MODEL_NAME --batch_size=1 --num_batches 1 --record-trajectories

print("\nDone. Files in output directory:")
for p in sorted(results_dir.iterdir()):
    print(" ", p.name)

Running generation...
MODELS_PROJECT_ROOT: /content/mattergen/mattergen
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mattergen/resolve/main/checkpoints/mattergen_base/checkpoints/last.ckpt "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mattergen/xet-read-token/ea430eab64b80855029c2941b9fda15f245a771a "HTTP/1.1 200 OK"
checkpoints/mattergen_base/checkpoints/l(…): 100% 461M/461M [00:04<00:00, 103MB/s]
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mattergen/resolve/main/checkpoints/mattergen_base/config.yaml "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/mattergen/ea430eab64b80855029c2941b9fda15f245a771a/checkpoints%2Fmattergen_base%2Fconfig.yaml "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/mattergen/ea430eab64b80855029c2941b9fda15f245a771a/checkpoints%2Fmattergen_base%2Fconf

## 5. Property-conditioned generation

Now we condition on an example target property. You can replace values/models with other available checkpoints.

> Tip: Start with moderate guidance (e.g., 1.5–2.5), then scan values to study quality-vs-adherence tradeoffs.


In [ ]:
from pathlib import Path

cond_results = Path("/content/results_conditioned")
cond_results.mkdir(exist_ok=True)

print("Running generation...")
!source .venv/bin/activate; export MODEL_NAME=dft_mag_density ; export RESULTS_PATH=/content/results_conditioned/ ; mattergen-generate $RESULTS_PATH --pretrained-name=$MODEL_NAME --batch_size=16 --properties_to_condition_on="{'dft_mag_density': 0.15}" --diffusion_guidance_factor=2.0

print("\nDone. Files in conditioned output directory:")
for p in sorted(cond_results.iterdir()):
    print(" -", p.name)

Running generation...
MODELS_PROJECT_ROOT: /content/mattergen/mattergen
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mattergen/resolve/main/checkpoints/dft_mag_density/checkpoints/last.ckpt "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mattergen/xet-read-token/ea430eab64b80855029c2941b9fda15f245a771a "HTTP/1.1 200 OK"
checkpoints/dft_mag_density/checkpoints/(…): 100% 512M/512M [00:06<00:00, 79.7MB/s]
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mattergen/resolve/main/checkpoints/dft_mag_density/config.yaml "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/mattergen/ea430eab64b80855029c2941b9fda15f245a771a/checkpoints%2Fdft_mag_density%2Fconfig.yaml "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/mattergen/ea430eab64b80855029c2941b9fda15f245a771a/checkpoints%2Fdft_mag_density%2

## 6. Parse generated outputs into Python objects

MatterGen writes structures as `.extxyz` and/or zipped CIF files. We'll load `.extxyz` with ASE and convert to Pymatgen for chemistry-aware analysis.


In [ ]:
# Install specific versions of dependencies to avoid conflicts, especially with numpy/scipy/ase/pymatgen
# Force re-installation to ensure consistency.
!source .venv/bin/activate; uv pip install --force-reinstall numpy==1.26.4 scipy==1.13.1 ase==3.25.0 pymatgen nglview

Using Python 3.12.13 environment at: /usr
Resolved 124 packages in 3.11s
Prepared 124 packages in 4.34s
Uninstalled 115 packages in 1.41s
Installed 124 packages in 419ms
 - anyio==4.12.1
 + anyio==4.13.0
 ~ argon2-cffi==25.1.0
 ~ argon2-cffi-bindings==25.1.0
 ~ arrow==1.4.0
 ~ ase==3.25.0
 + asttokens==3.0.1
 ~ async-lru==2.3.0
 - attrs==25.4.0
 + attrs==26.1.0
 ~ babel==2.18.0
 - beautifulsoup4==4.13.5
 + beautifulsoup4==4.14.3
 + bibtexparser==1.4.4
 ~ bleach==6.3.0
 ~ certifi==2026.2.25
 ~ cffi==2.0.0
 ~ charset-normalizer==3.4.6
 + comm==0.2.3
 ~ contourpy==1.3.3
 ~ cycler==0.12.1
 - debugpy==1.8.15
 + debugpy==1.8.20
 - decorator==4.4.2
 + decorator==5.2.1
 ~ defusedxml==0.7.1
 + executing==2.2.1
 ~ fastjsonschema==2.21.2
 ~ fonttools==4.62.1
 ~ fqdn==1.5.1
 ~ h11==0.16.0
 ~ httpcore==1.0.9
 ~ httpx==0.28.1
 ~ idna==3.11
 - ipykernel==6.17.1
 + ipykernel==7.2.0
 - ipython==7.34.0
 + ipython==9.11.0
 + ipython-pygments-lexers==1.1.1
 - ipywidgets==7.7.1
 + ipywidgets==8.1.8
 ~ isod

In [ ]:
from ase.io import read
from pymatgen.io.ase import AseAtomsAdaptor
import pandas as pd

extxyz_path = "results_unconditional/generated_crystals.extxyz"
atoms_list = read(extxyz_path, index=":")

structures = [AseAtomsAdaptor.get_structure(a) for a in atoms_list]

rows = []
for i, s in enumerate(structures):
    rows.append(
        {
            "sample_id": i,
            "formula": s.composition.reduced_formula,
            "n_sites": len(s),
            "volume": float(s.volume),
            "density": float(s.density),
        }
    )

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} structures")
df.head(10)


## 7. Quick statistical visualization

These plots help evaluate whether outputs look physically plausible and chemically diverse.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("talk")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["n_sites"], bins=15, kde=False, ax=axes[0])
axes[0].set_title("Distribution of atom counts per generated crystal")
axes[0].set_xlabel("Number of sites")

sns.histplot(df["volume"], bins=15, kde=False, ax=axes[1])
axes[1].set_title("Distribution of unit-cell volumes")
axes[1].set_xlabel("Volume (Å³)")

plt.tight_layout()
plt.show()


## 8. 3D visualization of a generated crystal

We'll render one generated structure using `py3Dmol` directly in notebook output.


In [ ]:
from ase.visualize import view
from ase.io import write

idx = 9

# Get the ASE Atoms object directly
atoms_to_view = read("unzipped_trajectories/gen_0.extxyz", ":")

# View atoms
view(atoms_to_view[0], viewer="x3d")

write("unzipped_trajectories/gen_0.gif", atoms_to_view[::10])

In [ ]:
from IPython.display import Image
Image(filename='unzipped_trajectories/gen_0.gif')

## 9. Exercises

1. **Guidance sweep:** Repeat conditioned generation with guidance factors `[0.0, 1.0, 2.0, 4.0]` and compare diversity/target adherence.
2. **Target sweep:** For a conditioned model, vary target values and inspect whether generated distributions shift in expected directions.
3. **Post-generation relaxation:** Use robust atomistic relaxation workflows before drawing thermodynamic conclusions.
4. **Novelty/uniqueness evaluation:** Use MatterGen evaluation tools to compute benchmark-style metrics.



## 10. Troubleshooting (Colab)

- **Out of memory:** reduce `--batch_size`.
- **Slow runtime:** ensure GPU runtime is selected.
- **Package conflicts:** restart runtime and rerun setup cells.
- **Checkpoint download delays:** first call to a checkpoint may take time while files are fetched.


# Task
Unzip "results_unconditional/generated_trajectories.zip" into a new directory, then load the trajectory from the unzipped .extxyz file into a list of ASE Atoms objects and visualize it.

## Unzip trajectory file

### Subtask:
Unzip 'results_unconditional/generated_trajectories.zip' into a new directory.


**Reasoning**:
First, I will create a new directory for the unzipped files to keep the file system organized. Then, I will unzip the specified archive into this new directory using a shell command.



In [ ]:
import os

unzip_dir = "unzipped_trajectories"
os.makedirs(unzip_dir, exist_ok=True)

zip_file_path = "results_unconditional/generated_trajectories.zip"
!unzip -o $zip_file_path -d $unzip_dir

print(f"Successfully unzipped {zip_file_path} to {unzip_dir}/")
print("Contents of unzipped_trajectories:")
!ls $unzip_dir

## Load and view trajectory

### Subtask:
Read the unzipped trajectory file (assuming it's an .extxyz file) into a list of ASE Atoms objects and then visualize it as a trajectory.


## Summary:

### Data Analysis Key Findings

*   A new directory named `unzipped_trajectories` was successfully created.
*   The `results_unconditional/generated_trajectories.zip` file was successfully unzipped into the `unzipped_trajectories` directory.
*   The unzipped directory contains a file named `gen_0.extxyz`.

### Insights or Next Steps

*   The next step should be to load the `gen_0.extxyz` file as an ASE Atoms object and proceed with its visualization as a trajectory.
